# Notebook 7 – Encoding Categorical Variables
**File:** `07_Categorical_Encoding.ipynb`

## Learning objectives
By the end of this notebook, you will understand categorical variables and choose an encoding method that fits the meaning of the data and the machine-learning model.

Topics:
- Nominal, ordinal, and binary variables
- Label, ordinal, one-hot, dummy, frequency, and target encoding
- Unknown categories and high-cardinality features
- Risks of inappropriate encoding

> **Important:** Encoding is not just converting words into numbers. The numbers must preserve the meaning of the category. Fit encoders on training data only to avoid data leakage.

## 1. What is Categorical Encoding?

Categorical encoding converts non-numeric categories into numeric representations that a machine-learning algorithm can process.

Examples:
- `color`: Red, Blue, Green
- `size`: Small, Medium, Large
- `payment_method`: Cash, Card, UPI

**Why is it needed?** Many ML algorithms require numeric input. A suitable encoding represents categories without introducing a false relationship between them.

## 2. Types of Categorical Variables

### Nominal variables
Categories have no natural order. Examples: city, color, browser, department.

### Ordinal variables
Categories have a meaningful order. Examples: Low, Medium, High; Bronze, Silver, Gold.

### Binary variables
Only two categories. Examples: Yes/No, True/False, Active/Inactive.

The type determines which encoding methods are reasonable.

In [1]:
import pandas as pd
import numpy as np

sample = pd.DataFrame({
    "city": ["Hyderabad", "Chennai", "Mumbai", "Hyderabad", "Chennai"],
    "size": ["Small", "Medium", "Large", "Medium", "Small"],
    "subscribed": ["Yes", "No", "Yes", "No", "Yes"],
    "purchased": [0, 1, 1, 0, 1]
})

display(sample)
print("Data types:")
print(sample.dtypes)

,city,size,subscribed,purchased
0,Hyderabad,Small,Yes,0
1,Chennai,Medium,No,1
2,Mumbai,Large,Yes,1
3,Hyderabad,Medium,No,0
4,Chennai,Small,Yes,1


Data types:
city            str
size            str
subscribed      str
purchased     int64
dtype: object


## 3. Label Encoding

Label encoding assigns an integer to each category, for example Red=0, Blue=1, Green=2.

**Use:** Commonly used for a classification target label (the value to predict). It can also be used for input features with models that treat category codes appropriately.

**Risk:** If used on a nominal input feature, numbers such as 0, 1, and 2 may suggest an order or distance that does not exist. For nominal input columns, one-hot encoding is often safer.

In [2]:
from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
sample["purchased_encoded"] = target_encoder.fit_transform(sample["purchased"])

print("Classes:", target_encoder.classes_)
display(sample[["purchased", "purchased_encoded"]])

Classes: [0 1]


,purchased,purchased_encoded
0,0,0
1,1,1
2,1,1
3,0,0
4,1,1


## 4. Ordinal Encoding

Ordinal encoding maps ordered categories to integers in the correct order.

Example: Small=0, Medium=1, Large=2.

**Use:** When the category order is meaningful and known. Define the order explicitly; do not rely on alphabetical order.

**Risk:** The numeric spacing may imply equal distances between levels. Small-to-Medium may not represent the same increase as Medium-to-Large.

In [3]:
from sklearn.preprocessing import OrdinalEncoder

size_order = [["Small", "Medium", "Large"]]
ordinal_encoder = OrdinalEncoder(
    categories=size_order,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

sample["size_encoded"] = ordinal_encoder.fit_transform(sample[["size"]]).astype(int)
display(sample[["size", "size_encoded"]])

,size,size_encoded
0,Small,0
1,Medium,1
2,Large,2
3,Medium,1
4,Small,0


## 5. One-Hot Encoding

One-hot encoding creates a separate 0/1 column for each category.

Example: a `city` feature becomes columns such as `city_Chennai`, `city_Hyderabad`, and `city_Mumbai`.

**Use:** Nominal input features when the number of unique categories is manageable. It avoids inventing an order between categories.

**Risk:** It can create many columns when a feature has thousands of unique categories (high cardinality).

In [4]:
one_hot = pd.get_dummies(sample["city"], prefix="city", dtype=int)
display(pd.concat([sample[["city"]], one_hot], axis=1))

,city,city_Chennai,city_Hyderabad,city_Mumbai
0,Hyderabad,0,1,0
1,Chennai,1,0,0
2,Mumbai,0,0,1
3,Hyderabad,0,1,0
4,Chennai,1,0,0


## 6. Dummy Variables

Dummy encoding is closely related to one-hot encoding. It often drops one category as a reference category, producing k−1 columns for k categories.

**Use:** Often useful in linear or regression models with an intercept, where dropping a reference category helps avoid perfect multicollinearity.

**Risk:** The omitted category becomes the baseline and must be understood when interpreting coefficients. For many tree-based workflows, keeping all one-hot columns can also be acceptable.

In [5]:
dummy_encoded = pd.get_dummies(
    sample["city"], prefix="city", drop_first=True, dtype=int
)
display(pd.concat([sample[["city"]], dummy_encoded], axis=1))

,city,city_Hyderabad,city_Mumbai
0,Hyderabad,1,0
1,Chennai,0,0
2,Mumbai,0,1
3,Hyderabad,1,0
4,Chennai,0,0


## 7. Frequency Encoding

Frequency encoding replaces each category with how often it appears in the training data. It may use counts or proportions.

**Use:** Sometimes useful for high-cardinality categories when frequency contains predictive information and a compact representation is needed.

**Risk:** Different categories can have the same frequency. Frequency does not represent category meaning. Calculate the mapping from training data only.

In [6]:
train_cities = pd.Series(["Hyderabad", "Chennai", "Hyderabad", "Mumbai", "Hyderabad"])
frequency_map = train_cities.value_counts(normalize=True).to_dict()

example_cities = pd.Series(["Hyderabad", "Chennai", "Mumbai"])
frequency_encoded = example_cities.map(frequency_map)

frequency_result = pd.DataFrame({
    "city": example_cities,
    "frequency": frequency_encoded
})
display(frequency_result)
print("Training frequency map:", frequency_map)

,city,frequency
0,Hyderabad,0.6
1,Chennai,0.2
2,Mumbai,0.2


Training frequency map: {'Hyderabad': 0.6, 'Chennai': 0.2, 'Mumbai': 0.2}


## 8. Target Encoding

Target encoding replaces each category with a statistic calculated from the target, commonly the target mean for regression or binary classification.

Example: encode each store by the historical average purchase rate for that store.

**Use:** Potentially useful for high-cardinality features when there is enough training data.

**Risk – data leakage and overfitting:** If a row's own target contributes to its encoded value, the model can indirectly see the answer. Use out-of-fold target encoding or a carefully regularized encoder. Never calculate target mappings using validation/test targets.

In [7]:
# Small illustrative example only. For real model training, use out-of-fold
# target encoding (or a library encoder designed to prevent leakage).
training_data = pd.DataFrame({
    "store": ["A", "A", "A", "B", "B", "C", "C", "C"],
    "purchased": [1, 1, 0, 0, 0, 1, 0, 1]
})

target_mean_map = training_data.groupby("store")["purchased"].mean()
training_data["store_target_mean_demo"] = training_data["store"].map(target_mean_map)

display(training_data)
print("Category target means (illustration):")
display(target_mean_map)

,store,purchased,store_target_mean_demo
0,A,1,0.666667
1,A,1,0.666667
2,A,0,0.666667
3,B,0,0.000000
4,B,0,0.000000
5,C,1,0.666667
6,C,0,0.666667
7,C,1,0.666667


Category target means (illustration):


store
A    0.666667
B    0.000000
C    0.666667
Name: purchased, dtype: float64

## 9. Handling Unknown Categories

An unknown category is a value that was not present when the encoder was fitted. For example, a model was trained with Hyderabad, Chennai, and Mumbai, but production data contains Pune.

Possible approaches:
- One-hot encoding with `handle_unknown="ignore"`: unseen values produce all-zero values for that encoded feature group.
- Ordinal encoding with a dedicated unknown code such as `-1`.
- Map rare or unseen values to an `Other` category, if that category is included during training.

Choose and test the behavior before deploying the model.

In [8]:
from sklearn.preprocessing import OneHotEncoder

city_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
city_encoder.fit(sample[["city"]])

known_and_unknown = pd.DataFrame({"city": ["Hyderabad", "Pune"]})
encoded_cities = city_encoder.transform(known_and_unknown[["city"]])

encoded_city_df = pd.DataFrame(
    encoded_cities,
    columns=city_encoder.get_feature_names_out(["city"])
)

display(pd.concat([known_and_unknown, encoded_city_df], axis=1))

,city,city_Chennai,city_Hyderabad,city_Mumbai
0,Hyderabad,0.0,1.0,0.0
1,Pune,0.0,0.0,0.0


## 10. High-Cardinality Categories

A high-cardinality feature has many unique categories.

Examples:
- Customer ID
- Product ID
- Postal code
- Website URL

Possible approaches:
- One-hot encoding for a small or moderate number of categories.
- Group rare categories into `Other` when it makes domain sense.
- Frequency encoding.
- Carefully validated target encoding.
- Feature engineering, such as extracting a meaningful region from a postal code.

**Special warning:** Unique identifiers such as customer ID are often not useful as ordinary categorical features. They may cause memorization and poor generalization.

In [9]:
products = pd.Series([
    "P100", "P101", "P102", "P100", "P103",
    "P104", "P100", "P105", "P106", "P107"
])

counts = products.value_counts()
rare_products = counts[counts < 2].index

products_grouped = products.where(~products.isin(rare_products), "Other")

print("Unique products before grouping:", products.nunique())
print("Unique categories after grouping:", products_grouped.nunique())
display(products_grouped.value_counts().rename_axis("product").reset_index(name="count"))

Unique products before grouping: 8
Unique categories after grouping: 2


,product,count
0,Other,7
1,P100,3


## 11. Compare Encoding Techniques

| Technique | Best suited for | Main risk |
|---|---|---|
| Label encoding | Target labels; selected model-compatible cases | False order for nominal inputs |
| Ordinal encoding | Ordered categories | Assumed order/spacing may be misleading |
| One-hot encoding | Nominal features with manageable cardinality | Many columns |
| Dummy encoding | Nominal features, often linear models | Reference category interpretation |
| Frequency encoding | Some high-cardinality features | Equal-frequency categories become indistinguishable |
| Target encoding | High-cardinality features with careful validation | Leakage and overfitting |

## 12. Risks of Inappropriate Encoding

1. **False order:** Encoding City A=0, City B=1, City C=2 can make a model treat cities as ordered.
2. **Data leakage:** Using target information from validation/test data or the same row's target can inflate scores.
3. **Too many columns:** One-hot encoding a unique ID can create a huge sparse feature matrix.
4. **Unknown values:** Production categories may not exist in the training data.
5. **Inconsistent mapping:** Rebuilding category mappings separately for training and prediction can assign different meanings to codes.
6. **Lost information:** Grouping rare categories may hide useful distinctions.

## 13. Fit Encoders on Training Data Only

Split the data first. Fit preprocessing only on the training set, then use the fitted transformer to transform validation/test/new data. A scikit-learn `Pipeline` or `ColumnTransformer` helps keep this process consistent.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

demo = pd.DataFrame({
    "city": ["Hyderabad", "Chennai", "Mumbai", "Hyderabad", "Chennai",
             "Mumbai", "Hyderabad", "Chennai", "Mumbai", "Hyderabad",
             "Chennai", "Mumbai"],
    "size": ["Small", "Medium", "Large", "Medium", "Small", "Large",
             "Small", "Large", "Medium", "Large", "Medium", "Small"],
    "purchased": [0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0]
})

X = demo[["city", "size"]]
y = demo["purchased"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("nominal", OneHotEncoder(handle_unknown="ignore"), ["city"]),
        ("ordinal", OrdinalEncoder(
            categories=[["Small", "Medium", "Large"]],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ), ["size"])
    ]
)

model_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

model_pipeline.fit(X_train, y_train)
print("Test accuracy:", round(model_pipeline.score(X_test, y_test), 3))

Test accuracy: 0.667


## 14. Business and AI/ML Use Cases

- **Retail:** Encode product category and customer membership level for purchase prediction.
- **Banking:** Encode payment type and employment category for risk modelling.
- **Healthcare:** Encode diagnosis groups carefully; category meaning and clinical context matter.
- **Customer churn:** One-hot encode manageable plan types; use validated approaches for many postal codes or product IDs.

## 15. Mini Practice Questions

1. What is categorical encoding?
2. What is the difference between nominal and ordinal variables?
3. Why can label encoding be risky for a nominal input feature?
4. When should ordinal encoding be used?
5. What is the difference between one-hot and dummy encoding?
6. Explain frequency encoding.
7. What is target encoding, and why can it cause data leakage?
8. How can unknown categories be handled?
9. What is a high-cardinality feature?
10. Why should an encoder be fitted on training data only?